### Model Experiments

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from src.visualization import silhouette_plot
from src.topic_analysis import get_top_words_per_topic
import pickle

In [0]:
df = spark.table("scientific_trend_analysis.core.arxiv_cleaned_sampled")
df_label = df.toPandas()

In [0]:
processed_text_list = [row['processed_text'] for row in df.select('processed_text').collect()]
print(f"Loaded {len(processed_text_list)} processed texts")

In [0]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=5, max_features=20000, sublinear_tf=True, stop_words='english', ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(processed_text_list)
feature_name = tfidf_vectorizer.get_feature_names_out()
print(feature_name)

selector = VarianceThreshold(threshold=0.0001)
X_tfidf_reduced = selector.fit_transform(X_tfidf)
print("Shape after tfidf: ", X_tfidf.shape)
print("Shape after tfidf and variance threshold:", X_tfidf_reduced.shape)

In [0]:
n_components = 15
svd = TruncatedSVD(n_components=n_components, random_state=2)

lsa_result = svd.fit_transform(X_tfidf_reduced)
lsa_result = normalize(lsa_result, norm='l2', axis=1)
print(f"LSA result shape: {lsa_result.shape}")

explained_variance = svd.explained_variance_ratio_.sum()
print(f"Total explained variance: {explained_variance * 100:.2f}%")

In [0]:
# Save LSA features for visualization (numpy array)
dbutils.fs.mkdirs("/Volumes/scientific_trend_analysis/core/model_artifacts/")
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_features.pkl", "wb") as f:
    pickle.dump(lsa_result, f)

#### K-Means

In [0]:
inertia = []
silhouette_scores = {}

k_range = range(6, 14)
for i in k_range:
    km = KMeans(n_clusters=i, 
                init='k-means++',
                n_init='auto',
                random_state=2,
                max_iter=100)
    km.fit(lsa_result)
    inertia.append(km.inertia_)
    cluster_labels = km.predict(lsa_result)
    
    score = silhouette_score(lsa_result, cluster_labels)
    silhouette_scores[i] = score
    print(f"k={i}, Silhouette Score: {score:.4f}")

plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of cluster', fontsize=14)
plt.ylabel('Inertia', fontsize=14)
plt.xticks(list(k_range))

# find the best k
best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k values according to Silhouette Score: {best_k}")


In [0]:
# final km model
final_kmeans = KMeans(n_clusters=best_k, random_state=2, n_init='auto')
cluster_labels_km = final_kmeans.fit_predict(lsa_result)

# Assign labels to the original dataframe
df_label['Cluster_ID_km'] = cluster_labels_km

silhouette_plot(lsa_result=lsa_result, cluster_labels=cluster_labels_km, best_k=best_k)

In [0]:
get_top_words_per_topic(df_label, 'Cluster_ID_km', 10)

In [0]:
display(df_label)

In [0]:
spark.createDataFrame(df_label).write.mode("overwrite") \
    .saveAsTable("scientific_trend_analysis.rep.arxiv_clustered")
print(f"Saved {len(df_label)} records with cluster assignments")